# $k\cdot p$ bulk bandstructure of Ge$_{1-x}$Sn$_{x}$

## 1. Setup general settings

### 1.1 Import modules

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import os
import numpy as np
from collections import defaultdict
import h5py
import matplotlib.pyplot as plt
from kpy30band import k_dot_p, process_data, plottings

### 1.2 Matplotlib settings

In [3]:
params = {'figure.figsize': (8, 6), 'legend.fontsize': 18, 'axes.labelsize': 24, 'axes.titlesize': 24,
          'xtick.labelsize':24, 'xtick.major.width':2, 'xtick.major.size':5, 'ytick.labelsize': 24,
          'ytick.major.width':2, 'ytick.major.size':5, 'xtick.minor.width':2, 'xtick.minor.size':3,
          'ytick.minor.width':2, 'ytick.minor.size':3, 'errorbar.capsize':2, 'lines.markersize':12,
          'lines.linewidth':2, 'lines.linestyle':'-'}
plt.rcParams.update(params)
plt.rc('font', size=24)
plt_lw=2 # line width for line plots

### 1.3 Set project id (easy to track)

In [4]:
my_project_id = 'kp_simulations' 
sample_id = 'GeSn' # Ge1-xSnx
sample_binaries = ['Sn', 'Ge']

### 1.4 Save data file path settings

In [5]:
save_f_dir = os.path.join(Path.home(),f'MyFolder/Projects/{my_project_id}/{sample_id}')
save_dataf_dir = f'{save_f_dir}/Data/'
save_figsf_dir = f'{save_f_dir}/Figs/'

os.makedirs(save_dataf_dir, exist_ok=True)
os.makedirs(save_figsf_dir, exist_ok=True)

In [6]:
save_figure = False
FigFormat = 'png'
FigDpi = 300

## 2. Set up simulation system information

### 2.1 Set $k$-path and compositions

In [30]:
# Which k-path to plot
kpath_list = ['L-G', 'G-X']
# Number of k-points in each section
nkpoints=41
# Whether to calculate both eigen values or eigenvectors of k.p Hamiltonian
calc_eigenval_only=True
# At which compositions to calculate
composition = [0, 0.1, 0.20, 1.0]

### 2.3 Run $k\cdot p$ calculations

#### 2.3.1 Initialize $k\cdot p$ Hamiltonian

In [31]:
use_this_params = {'Sn': {'ED_25l25u': 0}, 'Ge': {'ED_25l25u': 0.0}}

In [32]:
kp = k_dot_p(binaries=sample_binaries, pseudomorphic_strain=False, substrate=None, 
             alloy_crystal_structure='zb', use_this_params=use_this_params, 
             save_file_dir=save_dataf_dir)

#### 2.3.2 Calculate $k\cdot p$ energies at k-points

In [33]:
ek_data = kp.kp_30x30(compositions=composition, kpath_list=kpath_list, nkpoints=nkpoints, 
                      return_eigen_val_vec_both=not calc_eigenval_only,
                      save_data=True, save_file='ek_data.h5')

## 3. Post processing

### 3.1 Load saved data

In [34]:
process_data_cls = process_data(save_file_dir=save_dataf_dir) 

In [35]:
for_compos = process_data_cls.process_ek_data('ek_data.h5', read_this_k_paths=['L-G', 'G-X']) # Default: read all bands

In [44]:
band_ids = {'CB_id': 9, 'VBM_id': 8, 'VB2_id': 7, 'VB3_id': 4} # 

In [48]:
for compos, (_,bands_e,_) in for_compos.items():
    # Python indexting starts with 0
    CB_energy = bands_e[band_ids['CB_id'] -1] 
    VB_energy = bands_e[band_ids['VBM_id']-1]
    G_energy = bands_e[:, nkpoints-1]

    comp = float(compos) # composition
    print(f'Sample: {sample_binaries[0]}{comp:0.2f}{sample_binaries[1]}{1.0-comp:0.2f}')
    print(f'\tBandgap     = {min(CB_energy) - max(VB_energy) : 0.3f} eV') 
    print(f'\tBandgap (G) = {G_energy[band_ids['CB_id']-1] - G_energy[band_ids['VBM_id']-1]  : 0.3f} eV') 
    print(f'\tDeltaSO (G) = {G_energy[band_ids['VB2_id']-1] - G_energy[band_ids['VB3_id']-1]  : 0.3f} eV')
    print(f'\tE_L         = {CB_energy[0]  : 0.3f} eV')

Sample: Sn0.00Ge1.00
	Bandgap     =  0.684 eV
	Bandgap (G) =  0.814 eV
	DeltaSO (G) =  0.225 eV
	E_L         =  0.684 eV
Sample: Sn0.10Ge0.90
	Bandgap     =  0.490 eV
	Bandgap (G) =  0.490 eV
	DeltaSO (G) =  0.713 eV
	E_L         =  0.554 eV
Sample: Sn0.20Ge0.80
	Bandgap     =  0.212 eV
	Bandgap (G) =  0.212 eV
	DeltaSO (G) =  1.103 eV
	E_L         =  0.459 eV
Sample: Sn1.00Ge0.00
	Bandgap     =  0.000 eV
	Bandgap (G) =  0.000 eV
	DeltaSO (G) =  0.652 eV
	E_L         =  0.108 eV


## 4. Plottings

In [38]:
plt_kp = plottings(save_figure_dir=save_figsf_dir, log_info='1')

### 3.1 Plot band structure

In [28]:
plot_bands = None #None #[1,2,3,4,5,6]
set_energy_axis_limit = (-5, 5) #ymin, ymax

In [46]:
for compos, (kpts, bands_e, special_kpts) in for_compos.items():
    comp = float(compos) # composition
    # file name of the figure to save
    save_fig_file = f'{sample_binaries[0]}{comp:0.2f}{sample_binaries[1]}{1.0-comp:0.2f}_LGX.{FigFormat}' 
    #===========================================================================================
    fig, ax = plt_kp.plot(kpts, bands_e, ymin=-5, ymax=5, special_kpts=special_kpts, color=None, 
                          savefig=save_figure, save_file_name=save_fig_file, dpi=FigDpi)

* Plot saved: /home/Docs5/badal.mondal/linuxhome/MyFolder/Projects/kp_simulations/GeSn/Figs//Sn0.00Ge1.00_LGX.png
* Plot saved: /home/Docs5/badal.mondal/linuxhome/MyFolder/Projects/kp_simulations/GeSn/Figs//Sn0.10Ge0.90_LGX.png
* Plot saved: /home/Docs5/badal.mondal/linuxhome/MyFolder/Projects/kp_simulations/GeSn/Figs//Sn0.20Ge0.80_LGX.png
* Plot saved: /home/Docs5/badal.mondal/linuxhome/MyFolder/Projects/kp_simulations/GeSn/Figs//Sn1.00Ge0.00_LGX.png
